In [1]:
import os
import sys
import json
import numpy as np
import tensorflow as tf
sys.path.append('..')
from src.LSTM import LSTM
from src.metrics import evaluate_predictions

PROC_DIR = '../data/processed'
PRED_DIR = '../data/predictions'
REPORT_DIR = '../data/reports'
MODEL_DIR = '../data/models'
VOCAB_SIZE = 1001
ITEM_DIM = VOCAB_SIZE - 1
MAX_SEQ_LEN = 10
BATCH_SIZE = 256
MAX_EPOCHS = 30
PATIENCE = 3
SEED = 42
K_VALUES = [1, 3, 5, 9]
for d in [PRED_DIR, REPORT_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)
tf.random.set_seed(SEED)
np.random.seed(SEED)

I0000 00:00:1795169289.983579  129670 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795169290.012881  129670 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795169290.779046  129670 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)


E0000 00:00:1795169291.751873  129670 cuda_platform.cc:57] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_SYSTEM_DRIVER_MISMATCH: system has unsupported display driver / cuda driver combination
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:50

I0000 00:00:1795169291.751895  129670 cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
I0000 00:00:1795169291.751897  129670 cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: LOQ-15IRX9
I0000 00:00:1795169291.751899  129670 cuda_diagnostics.cc:183] hostname: LOQ-15IRX9
I0000 00:00:1795169291.751971  129670 cuda_diagnostics.cc:190] libcuda reported version is: 620.73.4
I0000 00:00:1795169291.751979  129670 cuda_diagnostics.cc:194] kernel reported version is: 620.70.6
E0000 00:00:1795169291.751980  129670 cuda_diagnostics.cc:287] k

In [2]:
X_padded = np.load(f'{PROC_DIR}/X_padded.npy').astype(np.uint8)
Y = np.load(f'{PROC_DIR}/y_target.npy').astype(np.float32)
splits = np.load(f'{PROC_DIR}/splits.npz')
train_ex, val_ex, test_ex = splits['train'], splits['validation'], splits['test']
users = test_ex[:, 0]
H_raw = X_padded[:, :, 1:]
S_full = np.concatenate([X_padded, Y[:, None, :].astype(np.uint8)], axis=1)
print(f"History without RARE column: {H_raw.shape}, full sequences: {S_full.shape}")

History without RARE column: (14074, 10, 1000), full sequences: (14074, 11, 1001)


In [3]:
def raw_window(u, p):
    L = min(p, MAX_SEQ_LEN)
    x = np.zeros((MAX_SEQ_LEN, ITEM_DIM), dtype=np.uint8)
    x[MAX_SEQ_LEN - L:] = H_raw[u, p - L:p]
    return x


def materialize(examples):
    X = np.stack([raw_window(u, p) for u, p in examples])
    y = S_full[examples[:, 0], examples[:, 1]].astype(np.float32)
    return X, y


Xtr, ytr = materialize(train_ex)
Xva, yva = materialize(val_ex)
Xte, yte = materialize(test_ex)
ytr_items = ytr[:, 1:]
yva_items = yva[:, 1:]
print(f"Train {Xtr.shape}, validation {Xva.shape}, test {Xte.shape}")

Train (38981, 10, 1000), validation (14074, 10, 1000), test (14074, 10, 1000)


In [4]:
def make_ds(X, y, shuffle):
    def gen():
        idx = np.random.permutation(len(X)) if shuffle else np.arange(len(X))
        for i in range(0, len(X), BATCH_SIZE):
            b = idx[i:i + BATCH_SIZE]
            yield X[b].astype(np.float32), y[b]
    spec = (
        tf.TensorSpec((None, MAX_SEQ_LEN, ITEM_DIM), tf.float32),
        tf.TensorSpec((None, ITEM_DIM), tf.float32),
    )
    return tf.data.Dataset.from_generator(gen, output_signature=spec).prefetch(tf.data.AUTOTUNE)


train_ds = make_ds(Xtr, ytr_items, shuffle=True)
val_ds = make_ds(Xva, yva_items, shuffle=False)
p = np.clip(ytr_items.mean(axis=0), 1e-6, 1 - 1e-6)
PRIOR_LOGIT = np.log(p / (1 - p)).astype(np.float32)


def to_full(scores):
    return np.concatenate([np.zeros((len(scores), 1), np.float32), scores], axis=1)

In [5]:
with open(f'{REPORT_DIR}/lstm_training_report.json') as f:
    cfg = json.load(f)['selected_config']
print(f"Using the RBM model's selected config: {cfg}")


class ValHitRatio(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.best = -1.0
        self.best_weights = None
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        scores = to_full(self.model.predict(Xva, batch_size=BATCH_SIZE, verbose=0))
        res = evaluate_predictions(scores, yva, K_VALUES)
        mhr = float(np.mean([res[k]['HitRatio'] for k in K_VALUES]))
        print(f"    epoch {epoch + 1:>2} | val mean HR={mhr:.4f}", flush=True)
        if mhr > self.best:
            self.best = mhr
            self.best_weights = self.model.get_weights()
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= PATIENCE:
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)


net =  LSTM(
    sequence_length=MAX_SEQ_LEN,
    input_dim=ITEM_DIM,
    output_dim=ITEM_DIM,
    hidden_units=cfg['hidden_units'],
    dropout=cfg['dropout'],
    learning_rate=cfg['learning_rate'],
)
net.model.layers[-1].bias.assign(PRIOR_LOGIT)
callbacks = [
    ValHitRatio(),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, min_lr=1e-5),
]
hist = net.model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=2,
)

Using the RBM model's selected config: {'learning_rate': 0.0003, 'dropout': 0.4, 'hidden_units': 64}
Epoch 1/30


/home/parth/Projects/BITS/DM/next-basket-prediction/.venv/lib/python3.14/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1795169292.880867  129670 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
E0000 00:00:1795169297.170531  129670 util.cc:131] oneDNN supports DT_UINT8 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


    epoch  1 | val mean HR=0.1543
153/153 - 5s - 35ms/step - loss: 0.0190 - val_loss: 0.0205 - learning_rate: 3.0000e-04
Epoch 2/30
    epoch  2 | val mean HR=0.1543
153/153 - 4s - 27ms/step - loss: 0.0190 - val_loss: 0.0205 - learning_rate: 3.0000e-04
Epoch 3/30
    epoch  3 | val mean HR=0.1542
153/153 - 4s - 27ms/step - loss: 0.0190 - val_loss: 0.0205 - learning_rate: 1.5000e-04
Epoch 4/30
    epoch  4 | val mean HR=0.1543
153/153 - 4s - 27ms/step - loss: 0.0190 - val_loss: 0.0205 - learning_rate: 7.5000e-05


In [8]:
raw_scores = to_full(net.model.predict(Xte, batch_size=BATCH_SIZE, verbose=0).astype(np.float32))
rbm_scores = np.load(f'{PRED_DIR}/lstm_test_scores.npy').astype(np.float32)
raw_res = evaluate_predictions(raw_scores, yte, K_VALUES)
rbm_res = evaluate_predictions(rbm_scores, yte, K_VALUES)
with open(f'{REPORT_DIR}/phase3_baseline_report.json') as f:
    base = json.load(f)

print(f"{'K':>3} | {'Model':<13} | {'HitRatio':>8} | {'Recall':>8}")
for k in K_VALUES:
    rows = [
        ('Raw history', raw_res[k]),
        ('RBM LSTM', rbm_res[k]),
        ('Most Popular', base['most_popular'][str(k)]),
        ('FP-Growth', base['fpgrowth'][str(k)]),
    ]
    for name, r in rows:
        print(f"{k:>3} | {name:<13} | {r['HitRatio']:>8.4f} | {r['Recall']:>8.4f}")

  K | Model         | HitRatio |   Recall
  1 | Raw history   |   0.0459 |   0.0131
  1 | RBM LSTM      |   0.1371 |   0.0602
  1 | Most Popular  |   0.1533 |   0.0707
  1 | FP-Growth     |   0.1533 |   0.0707
  3 | Raw history   |   0.1998 |   0.0927
  3 | RBM LSTM      |   0.1998 |   0.0927
  3 | Most Popular  |   0.1998 |   0.0927
  3 | FP-Growth     |   0.2034 |   0.0930
  5 | Raw history   |   0.2267 |   0.1032
  5 | RBM LSTM      |   0.2269 |   0.1030
  5 | Most Popular  |   0.2347 |   0.1044
  5 | FP-Growth     |   0.2374 |   0.1051
  9 | Raw history   |   0.2970 |   0.1353
  9 | RBM LSTM      |   0.3048 |   0.1387
  9 | Most Popular  |   0.2796 |   0.1219
  9 | FP-Growth     |   0.2829 |   0.1234


In [9]:
def variance_split(scores):
    s = scores[:, 1:]
    item_mean = s.mean(axis=0)
    total = s.var()
    return item_mean.var() / total, (s - item_mean).var() / total


for name, sc in [('Raw history', raw_scores), ('RBM LSTM', rbm_scores)]:
    item_share, user_share = variance_split(sc)
    print(f"{name}: item identity {item_share:.2%} | user {user_share:.2%}")


def hit_ratio(scores, y, k, mask):
    s = scores[mask].copy()
    s[:, 0] = -np.inf
    idx = np.argsort(-s, axis=1, kind='stable')[:, :k]
    yy = y[mask].copy()
    yy[:, 0] = 0
    hits = np.take_along_axis(yy, idx, axis=1).sum(axis=1) > 0
    valid = yy.sum(axis=1) > 0
    return float(hits[valid].mean())


mp = np.load(f'{PRED_DIR}/most_popular_scores.npy').astype(np.float32)
mp = mp if mp.shape[0] == len(users) else mp[users]
hist_len = (X_padded[users].sum(axis=2) > 0).sum(axis=1)
bins = [(1, 2), (3, 5), (6, 9), (10, 10)]
print(f"{'History':>8} | {'K':>3} | {'Raw':>8} | {'RBM':>8} | {'Most Pop':>8}")
for lo, hi in bins:
    m = (hist_len >= lo) & (hist_len <= hi)
    for k in [3, 9]:
        print(f"{lo}-{hi:>6} | {k:>3} | {hit_ratio(raw_scores, yte, k, m):>8.4f} | "
              f"{hit_ratio(rbm_scores, yte, k, m):>8.4f} | {hit_ratio(mp, yte, k, m):>8.4f}")

Raw history: item identity 99.99% | user 0.02%
RBM LSTM: item identity 99.28% | user 0.72%
 History |   K |      Raw |      RBM | Most Pop
1-     2 |   3 |   0.1574 |   0.1574 |   0.1574
1-     2 |   9 |   0.2507 |   0.2540 |   0.2449
3-     5 |   3 |   0.1946 |   0.1946 |   0.1946
3-     5 |   9 |   0.2924 |   0.2993 |   0.2769
6-     9 |   3 |   0.2578 |   0.2578 |   0.2578
6-     9 |   9 |   0.3520 |   0.3636 |   0.3298
10-    10 |   3 |   0.2212 |   0.2212 |   0.2212
10-    10 |   9 |   0.3284 |   0.3432 |   0.2890


In [10]:
np.save(f'{PRED_DIR}/raw_lstm_test_scores.npy', raw_scores)
net.model.save(f'{MODEL_DIR}/raw_lstm.keras')
with open(f'{REPORT_DIR}/raw_lstm_report.json', 'w') as f:
    json.dump({'config': cfg, 'test_raw_lstm': {str(k): v for k, v in raw_res.items()}}, f, indent=2)
print("Saved raw-history scores, model and report")

Saved raw-history scores, model and report
